# PaySim threshold analysis

Thresholds are calculated on the complete validation period only. The test set remains reserved for final evaluation.

In [1]:
import json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
metadata = json.loads((ROOT / 'models/fraud_model_v1.0.0_metadata.json').read_text(encoding='utf-8'))
table = pd.read_csv(ROOT / 'output/evaluation/threshold_analysis.csv')
{'rows': len(table), 'selected_threshold': metadata['threshold'], 'selection_data': 'validation only'}

{'rows': 91, 'selected_threshold': 0.31999999999999995, 'selection_data': 'validation only'}

## Best validation thresholds by F2

In [2]:
metrics = ['threshold','precision','recall','f2','fp','fn','alert_count','alerts_per_1000','fraud_amount_captured','fraud_amount_missed','fraud_amount_capture_rate']
table.sort_values(['f2','precision','threshold'], ascending=False)[metrics].head(15)

,threshold,precision,recall,f2,fp,fn,alert_count,alerts_per_1000,fraud_amount_captured,fraud_amount_missed,fraud_amount_capture_rate
27,0.32,0.996622,1.000000,0.999322,4,0,1184,6.194186,1.497277e+09,0.00000,1.000000
26,0.31,0.996622,1.000000,0.999322,4,0,1184,6.194186,1.497277e+09,0.00000,1.000000
25,0.30,0.996622,1.000000,0.999322,4,0,1184,6.194186,1.497277e+09,0.00000,1.000000
24,0.29,0.996622,1.000000,0.999322,4,0,1184,6.194186,1.497277e+09,0.00000,1.000000
23,0.28,0.996622,1.000000,0.999322,4,0,1184,6.194186,1.497277e+09,0.00000,1.000000
73,0.78,1.000000,0.999153,0.999322,0,1,1179,6.168028,1.496923e+09,353874.21875,0.999764
72,0.77,1.000000,0.999153,0.999322,0,1,1179,6.168028,1.496923e+09,353874.21875,0.999764
71,0.76,1.000000,0.999153,0.999322,0,1,1179,6.168028,1.496923e+09,353874.21875,0.999764
70,0.75,1.000000,0.999153,0.999322,0,1,1179,6.168028,1.496923e+09,353874.21875,0.999764
69,0.74,1.000000,0.999153,0.999322,0,1,1179,6.168028,1.496923e+09,353874.21875,0.999764


## Selected operating point and alternatives

In [3]:
candidate_thresholds = [0.20, 0.32, 0.50]
candidates = table.loc[table.threshold.round(2).isin(candidate_thresholds), metrics].copy()
candidates['selected'] = candidates.threshold.round(2).eq(round(metadata['threshold'], 2))
candidates

,threshold,precision,recall,f2,fp,fn,alert_count,alerts_per_1000,fraud_amount_captured,fraud_amount_missed,fraud_amount_capture_rate,selected
15,0.20,0.993266,1.000000,0.998646,8,0,1188,6.215112,1.497277e+09,0.00000,1.000000,False
27,0.32,0.996622,1.000000,0.999322,4,0,1184,6.194186,1.497277e+09,0.00000,1.000000,True
45,0.50,1.000000,0.999153,0.999322,0,1,1179,6.168028,1.496923e+09,353874.21875,0.999764,False


## Visual evidence

![Precision, recall and F2 by threshold](../output/evaluation/threshold_metrics.svg)

![Alert volume by threshold](../output/evaluation/threshold_alert_volume.svg)

## Recommendation and pending approval

The technical recommendation is **0.32** because it maximizes validation F2 while preserving 100% recall and fraud-amount capture, with 1,184 alerts (6.194 per 1,000 transactions). This is not a business-approved operating threshold. Approval remains contingent on confirmed FP/FN costs, daily investigation capacity and sign-off from the risk-policy owner.